# 🧩 0) Environment & Imports

In [1]:
# %% [markdown]
# # Sigma-only NeRF — Train / Resume / Evaluate
# This notebook:
# 1) loads a tiny scene,
# 2) builds SigmaMLP,
# 3) trains or resumes from a checkpoint,
# 4) (optionally) renders full opacity & sigma.
#
# Assumes your custom package layout:
#   nerflab.nerf_sigma_learning.{SigmaMLP, TrainCfg, train_sigma_only, ...}

# %% 
from pathlib import Path
import torch
from torch.amp import GradScaler, autocast

# --- your custom libs (keep paths as in your code) ---
from nerflab.nerf_sigma_learning import SigmaMLP, TrainCfg, train_sigma_only
from nerflab.nerf_sigma_learning.learning_utils import (
    count_params,
    load_model_weights,
    make_warmup_cosine_lambda,
    load_checkpoint_full,
)

# nerflab core
from nerflab import Camera
from nerflab.io import load_batch_simple, get_frame_ids_for_case
from nerflab.nerf_sigma_learning.ops.forward_sigma import nerf_opacity

# Optional: matplotlib for diagnostics
import matplotlib.pyplot as plt

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch: 2.7.1+cu128
CUDA available: True


# ⚙️ 1) Config Cell

In [2]:
# %% [markdown]
# ## Config

# %% 
# ---------- user config ----------
scene_dir = Path("../data/data_experiment2")
split = "train"
seed = 7
num_frames = 5
frame_offset = 0
K = 4096                     # rays per pose per batch (reduce on CPU if slow)
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

# ---------- model ----------
L_posenc = 8
hidden_dim = 64
depth = 4
density_bias = -1.0

# ---------- training ----------
steps = 5
print_every = 1
eval_every = 1
lr = 5e-4
weight_decay = 1e-4          # (fixed duplicate)
warmup = 500
grad_clip = 1.0
amp = True                   # AMP speeds up on CUDA; on CPU it's safely ignored
chunk_train = 32768
chunk_eval = 65536
eval_idx = 0

# ---------- checkpoints ----------
ckpt_best = scene_dir / "sigma_best.pt"
ckpt_last = scene_dir / "sigma_last.pt"

# ---------- resume options ----------
# If you want to start from scratch: set resume = None
resume = ckpt_best          # e.g., ckpt_best or ckpt_last; or None
reset_optim = False         # True = load weights only (fresh optimizer); False = full resume
extra_steps = 5             # If resuming: train for N *additional* steps; None = keep absolute cfg.steps
new_lr = None               # e.g., 2e-4 to override LR on resume
rewarm_steps = None         # e.g., 300 to rebuild scheduler warmup on resume

# RNG
rng = torch.Generator(device=device).manual_seed(seed)


# 📦 2) Load Data (one-time)

In [3]:
# %% [markdown]
# ## Load batch (images, poses)

# %% 
frame_ids, _ = get_frame_ids_for_case(
    scene_dir, split, seed=seed, num_frames=num_frames, frame_offset=frame_offset
)
batch = load_batch_simple(scene_dir, frame_ids)

images = batch["images"]          # (B, H, W) in [0,1]
H_wc   = batch["H_wc"]            # (B, 4, 4)
B, H, W = images.shape
images_flat = images.view(B, -1)

print(f"Loaded {B} frames at {H}x{W}")


Loaded 5 frames at 480x640


# 🔭 3) Ray Sampler (kept as your API)

In [4]:
# %% [markdown]
# ## Ray sampling helper (uses your nerflab Camera)

# %% 
cam = Camera(H_wc)

def get_batch_rays(debug: bool = False):
    """
    Returns:
      C_true: (B, R)
      delta : (B, R, N)
      pts   : (B, R, N, 3)
    """
    Osub, Dsub, idx_lin, uv_int = cam.get_rays_sampled(
        rays_per_pose=K, rng=rng, return_indices=True
    )
    t, delta, pts = cam.sample_along_rays(Osub, Dsub, rng=rng)
    C_true = torch.gather(images_flat, 1, idx_lin)

    if debug:
        print("subset shapes: O", Osub.shape, "D", Dsub.shape)
        print("sample shapes:", "t", t.shape, "delta", delta.shape, "pts", pts.shape)
        print("idx_lin:", idx_lin.shape, "| uv_int:", uv_int.shape)

    return C_true, delta, pts

# quick sanity (optional)
_ = get_batch_rays(debug=True)


subset shapes: O torch.Size([5, 4096, 3]) D torch.Size([5, 4096, 3])
sample shapes: t torch.Size([5, 4096, 40]) delta torch.Size([5, 4096, 40]) pts torch.Size([5, 4096, 40, 3])
idx_lin: torch.Size([5, 4096]) | uv_int: torch.Size([5, 4096, 2])


# 🧠 4) Build Model

In [5]:
# %% [markdown]
# ## Model

# %% 
model = SigmaMLP(
    L_posenc=L_posenc,
    hidden_dim=hidden_dim,
    depth=depth,
    skip_at=(2,),
    density_bias=density_bias,
)
print(f"SigmaMLP params: {count_params(model):,}")


SigmaMLP params: 14,977


# 🏋️ 5) Trainer Config

In [6]:
# %% [markdown]
# ## Train config (TrainCfg)

# %% 
cfg = TrainCfg(
    steps=steps,
    lr=lr,
    weight_decay=weight_decay,
    warmup_steps=warmup,
    grad_clip_norm=grad_clip,
    amp=amp,
    print_every=print_every,
    eval_every=eval_every,
    device=device,
    chunk_rays_train=chunk_train,
    chunk_rays_eval=chunk_eval,
    ckpt_best_path=str(ckpt_best),
    ckpt_last_path=str(ckpt_last),
    eval_idx=eval_idx,
)
cfg


TrainCfg(steps=5, lr=0.0005, weight_decay=0.0001, warmup_steps=500, grad_clip_norm=1.0, amp=True, print_every=1, eval_every=1, device='cpu', chunk_rays_train=32768, chunk_rays_eval=65536, ckpt_best_path='../data/data_experiment2/sigma_best.pt', ckpt_last_path='../data/data_experiment2/sigma_last.pt', eval_idx=0)

# 🔁 6) Resume Logic (weights-only or full)

In [7]:
# %% [markdown]
# ## Resume logic
# - reset_optim=False → full resume (model + optimizer + scheduler + scaler)
# - reset_optim=True  → load weights only (fresh optimizer/scheduler)
# - extra_steps: if set, extends cfg.steps to (resume_step + extra_steps)
# - new_lr, rewarm_steps: optional overrides on resume

# %% 
injected_optim = None
injected_sched = None
injected_scaler = None
start_step = 0

if (resume is not None) and Path(resume).exists():
    if reset_optim:
        # Load weights only
        _ = load_model_weights(model, str(resume), device)
        if extra_steps is not None:
            cfg.steps = int(extra_steps)
    else:
        # Full resume
        import torch.optim as optim_mod
        optim = optim_mod.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.99), weight_decay=cfg.weight_decay)
        sched = torch.optim.lr_scheduler.LambdaLR(
            optim, lr_lambda=make_warmup_cosine_lambda(cfg.steps, cfg.warmup_steps)
        )
        scaler = GradScaler("cuda", enabled=(cfg.amp and str(cfg.device).startswith("cuda")))

        ckpt = load_checkpoint_full(str(resume), model, optim, sched, scaler, device)
        ckpt_step = int(ckpt.get("step", 0))
        start_step = ckpt_step

        # LR override on resume
        if new_lr is not None:
            for g in optim.param_groups:
                g["lr"] = float(new_lr)

        # Rebuild scheduler with new warmup (if requested)
        if rewarm_steps is not None:
            new_sched = torch.optim.lr_scheduler.LambdaLR(
                optim, lr_lambda=make_warmup_cosine_lambda(cfg.steps, rewarm_steps)
            )
            sched = new_sched

        # Extend schedule by extra steps (if requested)
        if extra_steps is not None:
            cfg.steps = ckpt_step + int(extra_steps)

        injected_optim = optim
        injected_sched = sched
        injected_scaler = scaler

print(f"Resume from: {resume if (resume and Path(resume).exists()) else 'None'} | start_step={start_step} | total target steps={cfg.steps}")


Resumed from ../data/data_experiment2/sigma_best.pt (step=6, psnr=8.651703075821157)
Resume from: ../data/data_experiment2/sigma_best.pt | start_step=6 | total target steps=11


# 🚀 7) Train

In [8]:
# %% [markdown]
# ## Train / Continue training

# %% 
_ = train_sigma_only(
    model=model,
    get_batch_rays=get_batch_rays,
    H_wc=H_wc,
    images=images,
    rng=rng,
    cfg=cfg,
    optim=injected_optim,
    sched=injected_sched,
    scaler=injected_scaler,
    start_step=start_step,
)


[     7/11] train_loss=0.142126 | train_PSNR= 8.47 dB | lr=8.00e-06 | 0.6s
  → eval frame 0: MSE=0.136395, PSNR=8.65 dB
  ✓ saved BEST checkpoint: ../data/data_experiment2/sigma_best.pt (PSNR=8.65 dB)
[     8/11] train_loss=0.143265 | train_PSNR= 8.44 dB | lr=9.00e-06 | 5.6s
  → eval frame 0: MSE=0.136377, PSNR=8.65 dB
  ✓ saved BEST checkpoint: ../data/data_experiment2/sigma_best.pt (PSNR=8.65 dB)
[     9/11] train_loss=0.141387 | train_PSNR= 8.50 dB | lr=1.00e-05 | 10.6s
  → eval frame 0: MSE=0.136367, PSNR=8.65 dB
  ✓ saved BEST checkpoint: ../data/data_experiment2/sigma_best.pt (PSNR=8.65 dB)
[    10/11] train_loss=0.140817 | train_PSNR= 8.51 dB | lr=1.10e-05 | 15.5s
  → eval frame 0: MSE=0.136339, PSNR=8.65 dB
  ✓ saved BEST checkpoint: ../data/data_experiment2/sigma_best.pt (PSNR=8.65 dB)
[    11/11] train_loss=0.142659 | train_PSNR= 8.46 dB | lr=1.20e-05 | 20.6s
  → eval frame 0: MSE=0.136323, PSNR=8.65 dB
  ✓ saved LAST checkpoint: ../data/data_experiment2/sigma_last.pt
  ✓ sav